# Serve a Model as a REST API with FastAPI
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ajit-ai/Data_Science/blob/main/11_MLOps_Deployment/fastapi_model_serving.ipynb)

FastAPI wraps any Python function into a production HTTP API with automatic docs, validation (pydantic) and async support - the de-facto standard for ML serving.

We serve the iris model at `POST /predict`, test it locally without opening ports, then show Docker deployment.

In [ ]:
!pip install -q fastapi uvicorn scikit-learn nest_asyncio

## 1. Train + write main.py

In [ ]:
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
import pickle

X, y = load_iris(return_X_y=True)
pickle.dump(LogisticRegression(max_iter=1000).fit(X, y), open("iris_clf.pkl", "wb"))

In [ ]:
main_py = """
from fastapi import FastAPI
from pydantic import BaseModel, Field
import pickle

app = FastAPI(title="Iris Model API", version="1.0")
model = pickle.load(open("iris_clf.pkl", "rb"))

class IrisFeatures(BaseModel):
    sepal_length: float = Field(..., gt=0, lt=15)
    sepal_width: float = Field(..., gt=0, lt=10)
    petal_length: float = Field(..., gt=0, lt=12)
    petal_width: float = Field(..., gt=0, lt=8)

@app.get("/")
def health():
    return {"status": "ok"}

@app.post("/predict")
def predict(f: IrisFeatures):
    X = [[f.sepal_length, f.sepal_width, f.petal_length, f.petal_width]]
    pred = int(model.predict(X)[0])
    return {"species": ["setosa", "versicolor", "virginica"][pred]}
"""
with open("main.py", "w") as fh:
    fh.write(main_py)
print(main_py[:400])

## 2. Test the API in-process (no port needed)

In [ ]:
from fastapi.testclient import TestClient
import importlib, main
importlib.reload(main)
client = TestClient(main.app)

print(client.get("/").json())
r = client.post("/predict", json={
    "sepal_length": 5.1, "sepal_width": 3.5,
    "petal_length": 1.4, "petal_width": 0.2})
print(r.status_code, r.json())

bad = client.post("/predict", json={"sepal_length": -3})   # validation!
print(bad.status_code, "<- auto 422 from pydantic")

## 3. Real server + Docker deployment

In [ ]:
deploy_notes = """
# run for real:
uvicorn main:app --reload --port 8000
# interactive docs appear automatically at http://localhost:8000/docs

# Dockerfile:
FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
EXPOSE 8000
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]

# docker build -t iris-api . && docker run -p 8000:8000 iris-api
"""
print(deploy_notes)

**Production checklist**
- Version models (`/v1/predict`) + log every request/prediction.
- Load model once at startup (module level), never per request.
- Add auth (API key/OAuth), rate limits, and health probes for k8s.
- Latency matters: prefer ONNX/joblib over fat pickles; consider batching.